In [ ]:
#| default_exp eval

In [ ]:
#| export
from __future__ import annotations

import math
from collections.abc import Callable
from dataclasses import asdict, dataclass

import numpy as np
import torch
import torch.ao.nn.quantized as nnq
import torch.nn as nn
from torch.fx.passes.shape_prop import ShapeProp

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

An accuracy published alone is not comparable to anything. This harness keeps the **per-image** result, so
two models evaluated on the same images can be compared as a paired sample instead of two lonely numbers.

| Function | Criterion it answers |
|---|---|
| `correct_vector`, `wilson` | top-1: which images this model gets right, and how wide the interval around `k/n` is |
| `paired_delta` | top-1 again: is the difference with the reference distinguishable from zero |
| `params` | size: how many weights, packed quantized weights included |
| `peak_activation_bytes` | memory: the peak of live activations for one image |
| `macs` | compute: multiply-accumulates for one image |
| `agreement` | do two artifacts of the same model predict the same class (a parity check, not a card criterion) |

A NaN logit, an empty dataloader or a model left in training mode raises: a silent wrong number is worse
than a stack trace. The harness never moves the model, only the batches — the model must already be on
`device`.

In [ ]:
#| export
def _check_eval(model):
    "A model measured in training mode reports the batch it was given, not the model"
    if isinstance(model, nn.Module) and model.training:
        raise ValueError("model is in training mode: call model.eval() first")


def _run(model, dl, device):
    "One pass over `dl`: predicted classes and targets"
    _check_eval(model)
    preds, targets = [], []
    with torch.no_grad():
        for x, y in dl:
            out = model(x.to(device) if torch.is_tensor(x) else x)
            out = out if torch.is_tensor(out) else torch.as_tensor(np.asarray(out))
            if not torch.isfinite(out).all():
                raise ValueError("model returned logits that are not finite — evaluate a finite model")
            preds.append(out.detach().cpu().argmax(-1).numpy().reshape(-1))
            targets.append(np.asarray(y.detach().cpu() if torch.is_tensor(y) else y).reshape(-1))
    if not preds: raise ValueError("dataloader is empty — nothing to evaluate")
    return np.concatenate(preds), np.concatenate(targets)


def predictions(
    model: Callable,                     # a model, or anything callable on a batch of inputs
    dl,                                  # dataloader yielding (inputs, targets)
    device: str | torch.device = 'cpu',  # device the batches are moved to; the model must already live there
) -> np.ndarray:
    "Predicted class of every image, in dataloader order"
    return _run(model, dl, device)[0]


def correct_vector(
    model: Callable,                     # a model, or anything callable on a batch of inputs
    dl,                                  # dataloader yielding (inputs, targets)
    device: str | torch.device = 'cpu',  # device the batches are moved to; the model must already live there
) -> np.ndarray:
    "Per-image correctness, in dataloader order"
    preds, targets = _run(model, dl, device)
    return preds == targets

In [ ]:
show_doc(predictions)

In [ ]:
show_doc(correct_vector)

In [ ]:
#| export
def wilson(
    k: int,           # images classified correctly
    n: int,           # images evaluated
    z: float = 1.96,  # 1.96 for a 95 % interval
) -> tuple[float, float]:
    "Wilson score interval of an accuracy, as fractions"
    if n <= 0: raise ValueError("wilson needs n > 0 — pass the number of images evaluated")
    p, d = k / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0., centre - half), min(1., centre + half)

In [ ]:
show_doc(wilson)

In [ ]:
#| export
@dataclass(slots=True)
class PairedDelta:
    "Accuracy difference between two models evaluated on the same images, in percentage points"
    delta: float
    lo: float
    hi: float
    p_mcnemar: float
    n: int

    def as_dict(self) -> dict: return asdict(self)


def _pair(a, b):
    "Two per-image vectors read as arrays over the same images"
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape: raise ValueError(f"vectors must cover the same images, got {a.shape} and {b.shape}")
    if a.size == 0: raise ValueError("vectors are empty — nothing to compare")
    return a, b


def _mcnemar(n01, n10):
    "Exact two-sided McNemar p from the discordant counts"
    n = n01 + n10
    if n == 0: return 1.0
    return min(1., 2 * sum(math.comb(n, i) for i in range(min(n01, n10) + 1)) / 2 ** n)


def paired_delta(
    a: np.ndarray,       # per-image correctness of the reference
    b: np.ndarray,       # per-image correctness of the model compared to it
    n_boot: int = 2000,  # bootstrap resamples over the images
    seed: int = 0,       # seed of the resampling
) -> PairedDelta:
    "Accuracy difference b - a in points, with a paired bootstrap interval and the exact McNemar p"
    a, b = (x.astype(bool) for x in _pair(a, b))
    d, n = b.astype(np.int8) - a.astype(np.int8), a.size
    idx = np.random.default_rng(seed).integers(0, n, size=(n_boot, n))
    lo, hi = np.percentile(100 * d[idx].mean(1), [2.5, 97.5])
    return PairedDelta(float(100 * d.mean()), float(lo), float(hi),
                       _mcnemar(int((a & ~b).sum()), int((~a & b).sum())), n)

In [ ]:
show_doc(paired_delta)

In [ ]:
show_doc(PairedDelta)

In [ ]:
#| export
def agreement(
    pred_a: np.ndarray,  # predicted classes of one artifact
    pred_b: np.ndarray,  # predicted classes of the other
) -> float:
    "Fraction of images on which two artifacts predict the same class"
    a, b = _pair(pred_a, pred_b)
    return float((a == b).mean())

In [ ]:
show_doc(agreement)

In [ ]:
#| export
_CONV, _LINEAR = (nn.Conv2d, nnq.Conv2d), (nn.Linear, nnq.Linear)


def params(
    model: nn.Module,  # the model to weigh
) -> int:
    "Number of weights; a quantized module keeps its weight packed outside `parameters()`, so it is counted apart (its bias is not)"
    return (sum(p.numel() for p in model.parameters())
            + sum(m.weight().numel() for m in model.modules() if isinstance(m, (nnq.Conv2d, nnq.Linear))))


def macs(
    model: nn.Module,       # the model to count, in eval mode
    sample: torch.Tensor,   # a batch; only its first image is used
) -> int:
    "Multiply-accumulates of one forward at batch 1, over convolutions and linear layers only — normalisation, activations, pooling and additions are not counted"
    _check_eval(model)
    counted, handles = [], []
    def hook(m, inp, out):
        if isinstance(m, _LINEAR): counted.append(out.numel() * m.in_features)
        else:
            w = m.weight() if callable(m.weight) else m.weight   # a quantized module hands its weight back through a call
            counted.append(out.numel() * math.prod(w.shape[1:]))
    for m in model.modules():
        if isinstance(m, _CONV + _LINEAR): handles.append(m.register_forward_hook(hook))
    try:
        with torch.no_grad(): model(sample[:1])
    finally:
        for h in handles: h.remove()
    return sum(counted)


def _node_bytes(n):
    "Bytes of a node's output, 0 when it is not a tensor (TensorMetadata is itself a tuple, hence the `shape` test first)"
    meta = n.meta.get('tensor_meta')
    metas = [meta] if hasattr(meta, 'shape') else meta if isinstance(meta, (tuple, list)) else []
    return sum(math.prod(m.shape) * m.dtype.itemsize for m in metas if hasattr(m, 'shape'))


def _reuses_input(gm, n):
    "True when the node writes into its input's buffer instead of allocating one"
    if n.op == 'call_module': return getattr(gm.get_submodule(n.target), 'inplace', False)
    return str(getattr(n.target, '__name__', n.target)).endswith('_')


def peak_activation_bytes(
    model: nn.Module,      # the model to measure, in eval mode
    sample: torch.Tensor,  # a batch; only its first image is used
) -> int:
    "Peak bytes of the activations alive at once during one forward at batch 1; weights are size, not memory of work, so they are not counted"
    _check_eval(model)
    if isinstance(model, torch.fx.GraphModule): gm = model
    else:
        try: gm = torch.fx.symbolic_trace(model)
        except Exception as e:
            raise ValueError(f"cannot trace {type(model).__name__} ({e}) — pass an FX GraphModule, which is what "
                             f"`convert_fx` returns") from e
    ShapeProp(gm).propagate(sample[:1])
    nodes = list(gm.graph.nodes)
    last = {a: i for i, n in enumerate(nodes) for a in n.all_input_nodes}
    live, peak = {}, 0
    for i, n in enumerate(nodes):
        if n.op not in ('get_attr', 'output'):
            live[n] = (live.pop(n.all_input_nodes[0], _node_bytes(n)) if _reuses_input(gm, n) and n.all_input_nodes
                       else _node_bytes(n))
        peak = max(peak, sum(live.values()))
        for prev in n.all_input_nodes:
            if last.get(prev) == i: live.pop(prev, None)
    return peak

In [ ]:
show_doc(params)

In [ ]:
show_doc(macs)

In [ ]:
show_doc(peak_activation_bytes)

---

## Usage

```python
from fastermodels import correct_vector, wilson, paired_delta, predictions, agreement

ref = correct_vector(source_model, valid_dl)
opt = correct_vector(FasterModel.from_pretrained('artifact'), valid_dl)

k, n = int(opt.sum()), opt.size
wilson(k, n)                 # (0.9298, 0.9448) — the interval that belongs next to k/n
paired_delta(ref, opt)       # PairedDelta(delta=-0.51, lo=-1.2, hi=0.2, p_mcnemar=0.12, n=3925)
agreement(predictions(source_model, valid_dl), predictions(reloaded, valid_dl))
```

`paired_delta` compares the two vectors image by image, so it reads a small difference the two intervals
would leave undecided. `lo > floor` is the publication condition; `delta` alone is not.

The three other criteria are read off the model itself, for one image:

```python
from fastermodels import params, macs, peak_activation_bytes

params(fm)                             # 8_900_000 weights
macs(fm, sample)                       # multiply-accumulates for one image
peak_activation_bytes(fm, sample)      # bytes of activations alive at once
```

`macs` counts convolutions and linear layers, nothing else. `peak_activation_bytes` walks the FX graph and
keeps a live set, so an in-place operation reuses its input's buffer instead of adding one; a model that
cannot be traced raises rather than reporting 0.

---

## See Also

- [Model](00_model.html) - the model these numbers are measured on
- [Card](02_card.html) - where `k`, `n`, the interval and the delta are written down
- [Gate](03_gate.html) - the condition that reads `lo` against the floor

Tests live in `nbs/tests/test_eval.ipynb`.